> 在线讲义：[第 16 章 · 异步拷贝与多级流水线：搬下一块的时候算这一块](https://wikieden.github.io/cuda-to-tensor-core/lessons/16-async-copy-pipeline.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 16 章 · 异步拷贝与多级流水线：搬下一块的时候算这一块

本章回答：第 13 章的 tile 循环"搬 → 等 → 算 → 等"，搬的时候算力闲着、算的时候显存闲着，怎么让两件事同时进行？Ampere 的 `cp.async`（LDGSTS）比"读到寄存器再写 shared"好在哪？双缓冲、多级流水线怎么排？

配套讲义：仓库里 `cuda-programming/notes/lessons/16-async-copy-pipeline.md`（讲解、动画、练习都在讲义里）。对应原文：PG §3.2.2.3 Asynchronous Execution Features（`materials/text/cuda-programming-guide.txt` 第 8754–8865 行）、§4.11 Pipelines（第 18584–18958 行）、§4.12.1 Using LDGSTS 与 §4.12.1.2 Prefetching Data（第 18967–19532 行）；Best Practices §10.2.3.4（`text/cuda-c-best-practices-guide.txt` 第 2041–2143 行）；PTX §9.7.10.28.3 cp.async（`text/ptx-isa-9.4-focus-mma-wgmma-tcgen05-tma.txt` 第 2227–2486 行）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 搬 4 算 4 | 8 × (4 + 4) vs 4 + 8 × 4 | `pipeline(8, 4, 4, 1)` / `pipeline(8, 4, 4, 2)` | 64 vs 36 |
| 第四步 搬 4 算 2 | 同步 / 双缓冲 / 3 级 | `pipeline(8, 4, 2, s)` | 48、34、34 |
| 第四步 算比搬慢 | ≈ 算的时间 | `pipeline(8, 2, 4, 2)` | 34 = 32 + 第一次搬 2 |
| 第四步 S 级 buffer | `s * block.size()` | `stage_offset(s, 256)` | 0、256、512 |
| 第五步 相位 | 到齐后翻转、重置 | `Barrier(4)`：3 次 `arrive` 后 `done` 为 False，第 4 次后 True | 相位 0 → 1，计数回到 4 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**类 `Barrier`**（来自 `ch16_async_pipeline.py`）：异步 barrier（PG §4.10）：期望 expected 次 arrive；到齐后相位翻转、计数重置。

In [ ]:
class Barrier:
    """异步 barrier（PG §4.10）：期望 expected 次 arrive；到齐后相位翻转、计数重置。"""

    def __init__(self, expected: int):
        self.expected = expected
        self.count = expected
        self.phase = 0

    def arrive(self) -> int:
        """报到，不停下；返回 token（当前相位）。"""
        token = self.phase
        self.count -= 1
        if self.count == 0:
            self.phase ^= 1
            self.count = self.expected
        return token

    def done(self, token: int) -> bool:
        """wait(token) 能不能通过：token 那一相位已经结束（相位翻过去了）。"""
        return self.phase != token

**函数 `pipeline`**（来自 `ch16_async_pipeline.py`）：n 块数据，搬一块 t_load、算一块 t_compute，shared 里有 stages 份 buffer。

In [ ]:
def pipeline(n: int, t_load: float, t_compute: float, stages: int) -> tuple[float, list, list]:
    """n 块数据，搬一块 t_load、算一块 t_compute，shared 里有 stages 份 buffer。

    规则：搬运一块接一块（一条显存通道）；第 i 块要等第 i − stages 块算完、buffer 空出来才能开始搬；
    计算一块接一块，第 i 块要等它搬完。stages = 1 就是"搬完再算"的同步写法。
    返回（总时间, 每块的搬运时段, 每块的计算时段）。
    """
    loads, computes = [], []
    for i in range(n):
        start = loads[-1][1] if loads else 0
        if i >= stages:
            start = max(start, computes[i - stages][1])     # 等 buffer 空
        loads.append((start, start + t_load))
        c0 = max(loads[i][1], computes[-1][1] if computes else 0)
        computes.append((c0, c0 + t_compute))
    return computes[-1][1], loads, computes

**函数 `stage_offset`**（来自 `ch16_async_pipeline.py`）：PG §4.12.1.2 的 prefetch_kernel：第 s 份 buffer 从 s * block.size() 开始。

In [ ]:
def stage_offset(stage: int, block_size: int) -> int:
    """PG §4.12.1.2 的 prefetch_kernel：第 s 份 buffer 从 s * block.size() 开始。"""
    return stage * block_size

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 搬 4、算 4，8 块")
print("  同步（搬完再算）：", pipeline(8, 4, 4, 1)[0])
print("  双缓冲：", pipeline(8, 4, 4, 2)[0], "（≈ 4 + 8 × 4）")

In [ ]:
print("\n第四步 搬 4、算 2，8 块")
for s, name in ((1, "同步"), (2, "双缓冲"), (3, "3 级")):
    print(f"  {name:<4}: {pipeline(8, 4, 2, s)[0]}")
_, loads, _ = pipeline(8, 4, 2, 2)
print("  双缓冲的搬运首尾相接：", loads[:3], "…")
print("  max(搬, 算) × n =", max(4, 2) * 8, "，再加上最后一块的计算 2")

In [ ]:
print("\n第四步 反过来：算比搬慢（搬 2、算 4，8 块）")
print("  同步：", pipeline(8, 2, 4, 1)[0], "；双缓冲：", pipeline(8, 2, 4, 2)[0], "≈ 算的时间 32 + 第一次搬 2")

In [ ]:
print("\n第四步 S 级 buffer 的起点（block.size() = 256）")
print("  ", [stage_offset(s, 256) for s in range(3)], "；轮转：", [(k + 1) % 3 for k in range(5)])

In [ ]:
print("\n第五步 barrier 的相位：期望 4 次 arrive")
bar = Barrier(4)
tokens = [bar.arrive() for _ in range(3)]
print("  3 个线程报到后：相位", bar.phase, "，wait 能过吗？", bar.done(tokens[0]))
bar.arrive()
print("  第 4 个报到后：相位", bar.phase, "，wait 能过吗？", bar.done(tokens[0]), "；计数重置为", bar.count)

## 3. 代码里才看得到的

1. **`stages = 1` 就是同步写法**：只有一份 buffer，搬第 i 块要等第 i − 1 块算完，搬和算完全串行。
2. **模型里只有一条显存通道**：搬运一块接一块，所以"搬是瓶颈"时多加 buffer 也快不了；真实 GPU 上多个 block 一起搬，但每个 block 的规律相同。
3. **`Barrier.done(token)` 看的是相位有没有翻过去**：和 `cuda::barrier::wait(token)` 一样，token 记下的是 arrive 时的相位。

## 4. 自己改一改

- 把 `pipeline` 的搬运时间改成"不稳定"（比如每块 2 或 6 交替），比较 2 级和 4 级的总时间——这就是讲义说要多于 2 级的原因。
- 打印 `pipeline(8, 4, 4, 2)` 的 `loads` 和 `computes`，画成时间线看搬和算怎么错开。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 搬完再算两头闲，搬这块时算上块；
> cp.async 下单直送台面：不经寄存器、发了就走，搬完只告诉自己，别人要看还得同步；
> 双缓冲两份台面轮着用，多级流水多几块在路上；
> 流水线只能藏起快的那一步，慢的那一步决定总时间；
> barrier 有相位：先报到、做别的、再等。